# Analysis over data preparation

After cleaning each dataset from undesired events and adding new engineered features, we mimik the analyses done in the data understanding phase.

Below are defined the functions later called for each dataset:
- `_load_dataset(*)` loads the `.root` files and returns a Pandas DataFrame;
- `_correlation_heatmap(*)` shows the Pearson and Spearman correlation heatmap between continuous features only;
- `_scatterplot_MET(*)` shows the relationship between the features `MET_pt` and `GenMET_pt`;
- `_data_distributions(*)` shows the distributions via histograms of the continuous features;
- `_boxplots(*)` shows the boxplot for each continuous feature to indentify outliers;
- `_data_barplots(*)` shows the barplots for each descrete feature.
- `nJet_vs_nGenJet` shows the barplots for the reconstructed and generator level jet multiplicities side by side. This time is defined separately from the other barplots as the jet features are only present in the HToAATo2Mu2B dataset.

Finally, some features are dropped as they are not considered relevant for the downstream ML model.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import uproot

PALETTE = "royalblue"

TREE_NAME = "Events"

/home/matilde/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
def _load_dataset_auto(path, tree_name):
    with uproot.open(f"{path}:{tree_name}") as tree:
        scalar_branches = []
        for name in tree.keys():
            try:
                interp = tree[name].interpretation
                dtype = interp.numpy_dtype
                if dtype.shape == ():
                    scalar_branches.append(name)
            except Exception:
                pass

        df = tree.arrays(scalar_branches, library="pd")

    float_features = df.select_dtypes(include=["float32", "float64"]).columns.tolist()
    int_features   = df.select_dtypes(include=["int32", "int64", "uint32", "uint64"]).columns.tolist()

    df[float_features] = df[float_features].astype(np.float64)
    df[int_features]   = df[int_features].astype(np.int64)

    print(f"  Auto-detected {len(float_features)} float features, {len(int_features)} int features")
    return df, float_features, int_features

In [3]:
def _load_dataset(FLOAT_FEATURES, INT_FEATURES, path, tree_name):
    """
        Loads branches from a .root file and returns a Pandas DataFrame.
    """
    all_branches = FLOAT_FEATURES + INT_FEATURES
    with uproot.open(f"{path}:{tree_name}") as tree:
        df = tree.arrays(all_branches, library="pd")

    # Explicit casting
    df[FLOAT_FEATURES] = df[FLOAT_FEATURES].astype(np.float64)
    df[INT_FEATURES]   = df[INT_FEATURES].astype(np.int64)
    return df

In [4]:
def _correlation_heatmap(df, FLOAT_FEATURES, label, method="pearson"):
    corr = df[FLOAT_FEATURES].replace([np.inf, -np.inf], np.nan).corr(method=method)

    print(f"\nCouples of features with {method} correlation > 0.70:")
    high_corr = []
    for i in range(len(corr.columns)):
        for j in range(i):
            corr_value = corr.iloc[i, j]
            if abs(corr_value) > 0.70:
                feat1 = corr.columns[i]
                feat2 = corr.columns[j]
                high_corr.append((feat1, feat2, corr_value))
                print(f"{feat1}, {feat2} = {corr_value:.2f}")

    if "GenMET_pt" in corr.columns:
        print(f"\n{method.capitalize()} correlation with GenMET_pt:")
        genmet_corr = (
            corr["GenMET_pt"]
            .drop("GenMET_pt")
            .sort_values(key=abs, ascending=False)
        )
        for feat, val in genmet_corr.items():
            print(f"    {feat:<35} {val:+.3f}")
            
    n = len(corr)

    fig_size  = max(14, n * 0.45)
    font_size = max(5, min(8, 250 // n))

    fig, ax = plt.subplots(figsize=(fig_size, fig_size * 0.9))
    sns.heatmap(
        corr,
        annot=True, fmt=".2f",
        cmap="coolwarm", linewidths=0.3, linecolor="gray",
        annot_kws={"size": font_size},
        ax=ax
    )
    ax.set_title(f"{method} correlation heatmap — {label}", fontsize=12, fontweight="bold", pad=12)
    ax.tick_params(axis="x", rotation=45, labelsize=font_size+1)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=font_size+1)
    ax.tick_params(axis="y", rotation=0,  labelsize=font_size+1)
    fig.tight_layout()
    plt.show()
    return

In [5]:
def _scatterplot_MET(df, label, color):
    x = df["MET_pt"]
    y = df["GenMET_pt"]
    mask = np.isfinite(x) & np.isfinite(y)

    # Statistics
    print(f"Statistics for dataset {label}:")
    print(f"    Mean MET_pt: {x.mean()}, Median MET_pt: {x.median()}")
    print(f"    Mean GenMET_pt: {y.mean()}, Median GenMET_pt: {y.median()}")

    plt.figure(figsize=(10, 8))
    plt.scatter(
        x[mask], y[mask], s=4, alpha=0.35,
        color=color, label=label, rasterized=True)

    # y = x line
    lim_max = plt.xlim()[1]
    lim = [0, lim_max]
    plt.plot(lim, lim, "k--", lw=1, label="y = x")

    plt.title(f"GenMET vs MET - {label}", fontsize=13, fontweight="bold")
    plt.xlabel("MET_pt [GeV]", fontsize=11)
    plt.ylabel("GenMET_pt [GeV]", fontsize=11)
    plt.legend(markerscale=3, fontsize=9)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.plot()
    return

In [6]:
def _data_distributions(df, FLOAT_FEATURES, label, color):
    n_feat = len(FLOAT_FEATURES)
    ncols = 3
    nrows = (n_feat + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = axes.flatten()

    for idx, feat in enumerate(FLOAT_FEATURES):
        ax = axes[idx]
        vals = df[feat].replace([np.inf, -np.inf], np.nan).dropna()
        ax.hist(vals, bins=100, density=True, histtype="step",
                linewidth=1.5, color=color, label=label)
        ax.set_xlabel(feat, fontsize=9)
        ax.set_ylabel("Density", fontsize=8)
        ax.set_title(feat, fontsize=9, fontweight="bold")
        ax.legend(fontsize=7)
        ax.grid(alpha=0.25)

    for j in range(idx + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(f"Distributions for float features - {label}", fontsize=14, fontweight="bold", y=1.01)
    fig.tight_layout()
    plt.show()
    return

In [7]:
def _boxplots(df, FLOAT_FEATURES, label, color):
    n_feat = len(FLOAT_FEATURES)
    ncols = 3
    nrows = (n_feat + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = axes.flatten()

    print(f"Outliers — {label}")

    for idx, feat in enumerate(FLOAT_FEATURES):
        ax = axes[idx]
        vals = df[feat].replace([np.inf, -np.inf], np.nan).dropna()

        # IQR
        q1, q3 = vals.quantile(0.25), vals.quantile(0.75)
        IQR = q3 - q1
        lower, upper = q1 - 1.5 * IQR, q3 + 1.5 * IQR
        n_outliers = ((vals < lower) | (vals > upper)).sum()
        pct = n_outliers / len(vals) * 100
        print(f"  {feat:<20} {n_outliers:>6} outliers ({pct:.2f}%)")

        bp = ax.boxplot(vals, patch_artist=True, vert=True,
                        flierprops=dict(marker=".", markersize=2, alpha=0.3,
                                        markerfacecolor=color, markeredgecolor=color),
                        medianprops=dict(color="black", linewidth=1.5),
                        boxprops=dict(facecolor=color, alpha=0.5),
                        whiskerprops=dict(color=color),
                        capprops=dict(color=color))

        ax.set_title(feat, fontsize=9, fontweight="bold")
        ax.set_ylabel(feat, fontsize=8)
        ax.set_xticks([])
        ax.text(0.97, 0.97, f"{n_outliers} outliers\n({pct:.1f}%)",
                transform=ax.transAxes, fontsize=7,
                verticalalignment="top", horizontalalignment="right",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))
        ax.grid(axis="y", alpha=0.25)

    for j in range(idx + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(f"Boxplots — {label}", fontsize=14, fontweight="bold", y=1.01)
    fig.tight_layout()
    plt.show()
    return

In [8]:
def _draw_bar(ax, df, feat, color, label):
    vals = df[feat].dropna()
    bins = np.arange(int(vals.min()), int(vals.max()) + 2) - 0.5
    centers = (bins[:-1] + bins[1:]) / 2
    counts, _ = np.histogram(vals, bins=bins)
    ax.bar(centers, counts, width=0.8, color=color, label=label, alpha=0.85)
    ax.set_xlabel(feat, fontsize=10)
    ax.set_ylabel("Counts", fontsize=10)
    ax.set_title(feat, fontsize=11, fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)
    return

def _data_barplots(df, INT_FEATURES, label, color):
    n_feat = len(INT_FEATURES)
    ncols = 2
    nrows = (n_feat + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 5 * nrows))
    axes = axes.flatten()

    for idx, feat in enumerate(INT_FEATURES):
        _draw_bar(axes[idx], df, feat, color, label)

    for j in range(idx + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(f"Barplots for integer features — {label}", fontsize=14, fontweight="bold", y=1.01)
    fig.tight_layout()
    plt.show()
    return

---

## Unified dataset analysis

In [9]:
df, float_features, int_features = _load_dataset_auto("../TrainingDataset/training.root", TREE_NAME)

print(f"    {df.shape[1]} features x {df.shape[0]} events")
print(f"    Number of float features: {len(float_features)}")
print(f"    Number of int features:   {len(int_features)}")

  Auto-detected 44 float features, 4 int features
    48 features x 2742601 events
    Number of float features: 44
    Number of int features:   4


In [13]:

df.to_parquet("../TrainingDataset/training.parquet", index=False)